# 02 — Forecasting Lab

Let's build and compare forecasting models interactively.
You can change parameters and see how performance changes.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

df = pd.read_csv('../data/smart_meter_data.csv', parse_dates=['timestamp'])

# Add cyclical features
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

features = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'temperature_c']
X = df[features]
y = df['load_kw']

# Time-based split
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

### Tune the Random Forest

Change `n_estimators` and `max_depth` below and re-run.

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
pred = rf.predict(X_test)
print(f"Random Forest MAE: {mean_absolute_error(y_test, pred):.4f}")

### Compare with Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print(f"Linear Regression MAE: {mean_absolute_error(y_test, pred_lr):.4f}")

### Naive baseline

In [ ]:
lag = 24 * 4
baseline = df['load_kw'].shift(lag).iloc[split:]
valid = ~baseline.isna()
print(f"Naive baseline MAE: {mean_absolute_error(y_test[valid], baseline[valid]):.4f}")

### Feature importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
importances.plot(kind='bar', title='Feature Importance')
importances

### Your turn

1. Try `n_estimators=10`. How much does MAE increase?
2. Try `max_depth=3`. What happens?
3. Add a new feature: hour squared (to capture non-linear time effects).
   Does it help?
4. Remove `temperature_c` from the feature list. How much worse does
   the model get?